##Data Cleaning and Transformation

In [0]:
from pyspark.sql import SparkSession

spark = SparkSession.builder\
    .appName("OlistData")\
        .getOrCreate()

In [0]:
#connect ADLSgen2 to Databricks

spark.conf.set("fs.azure.account.key.adlsgen2spark01.dfs.core.windows.net",
                "Storageaccount -> Security + Networking -> Access Keys-->Key1valuePaste here ")

In [0]:
#basePath
adlsgen2CSVpath = "abfss://olist-data@adlsgen2spark01.dfs.core.windows.net/csv/"
adlsgen2PARQUETpath = "abfss://olist-data@adlsgen2spark01.dfs.core.windows.net/parquet/"

In [0]:
customers_df = spark.read.csv(adlsgen2CSVpath + "olist_customers_dataset", header=True, inferSchema=True) 
geolocation_df = spark.read.csv(adlsgen2CSVpath + "olist_geolocation_dataset", header=True, inferSchema=True) 
order_items_df = spark.read.csv(adlsgen2CSVpath + "olist_order_items_dataset", header=True, inferSchema=True) 
payments_df = spark.read.csv(adlsgen2CSVpath + "olist_order_payments_dataset", header=True, inferSchema=True) 
reviews_df = spark.read.csv(adlsgen2CSVpath + "olist_order_reviews_dataset", header=True, inferSchema=True) 
orders_df = spark.read.csv(adlsgen2CSVpath + "olist_orders_dataset", header=True, inferSchema=True) 
products_df = spark.read.csv(adlsgen2CSVpath + "olist_products_dataset", header=True, inferSchema=True) 
sellers_df = spark.read.csv(adlsgen2CSVpath + "olist_sellers_dataset", header=True, inferSchema=True) 
catgeory_translation_df = spark.read.csv(adlsgen2CSVpath + "product_category_name_translation", header=True, inferSchema=True) 

In [0]:
customers_df.show(5)

+--------------------+--------------------+------------------------+--------------------+--------------+
|         customer_id|  customer_unique_id|customer_zip_code_prefix|       customer_city|customer_state|
+--------------------+--------------------+------------------------+--------------------+--------------+
|06b8999e2fba1a1fb...|861eff4711a542e4b...|                   14409|              franca|            SP|
|18955e83d337fd6b2...|290c77bc529b7ac93...|                    9790|sao bernardo do c...|            SP|
|4e7b3e00288586ebd...|060e732b5b29e8181...|                    1151|           sao paulo|            SP|
|b2b6027bc5c5109e5...|259dac757896d24d7...|                    8775|     mogi das cruzes|            SP|
|4f2d8ab171c80ec83...|345ecd01c38d18a90...|                   13056|            campinas|            SP|
+--------------------+--------------------+------------------------+--------------------+--------------+
only showing top 5 rows


In [0]:
#Identify missing values

In [0]:
from pyspark.sql.functions import *
    

In [0]:
def missing_values(df,df_name):
    print(f'Missing values in {df_name}:')
    df.select([count(when(col(c).isNull(),c)).alias(c) for c in df.columns]).show()

In [0]:
missing_values(customers_df,'customers_df')
missing_values(geolocation_df,'geolocation_df')
missing_values(order_items_df,'order_items_df')
missing_values(payments_df,'order_payments_df')
missing_values(reviews_df,'order_reviews_df')
missing_values(orders_df,'orders_df')
missing_values(products_df,'products_df')
missing_values(sellers_df,'sellers_df')
missing_values(catgeory_translation_df,'catgeory_translation_df')

Missing values in customers_df:
+-----------+------------------+------------------------+-------------+--------------+
|customer_id|customer_unique_id|customer_zip_code_prefix|customer_city|customer_state|
+-----------+------------------+------------------------+-------------+--------------+
|          0|                 0|                       0|            0|             0|
+-----------+------------------+------------------------+-------------+--------------+

Missing values in geolocation_df:
+---------------------------+---------------+---------------+----------------+-----------------+
|geolocation_zip_code_prefix|geolocation_lat|geolocation_lng|geolocation_city|geolocation_state|
+---------------------------+---------------+---------------+----------------+-----------------+
|                          0|              0|              0|               0|                0|
+---------------------------+---------------+---------------+----------------+-----------------+

Missing valu

# Handle Missing values

1. Drop missing Values (for critical columns)
2. Fill missing values (for numerical columns) # ordinal , catgeorical  -- read one on these handling missing values
3. Impute missing values ( for continuous values)  # graph mai continous
#terms coming from Datasience

In [0]:
orders_df_cleaned = orders_df.na.drop(subset=['order_id','customer_id','order_status'])
#so how can we have a order without an orderid, customerid, orderstatus = so if tehse are not there m,eans we need to drop as means we cannot consider that data
#so these col - gives power to drop the data
orders_df_cleaned.show()


+--------------------+--------------------+------------+------------------------+-------------------+----------------------------+-----------------------------+-----------------------------+
|            order_id|         customer_id|order_status|order_purchase_timestamp|  order_approved_at|order_delivered_carrier_date|order_delivered_customer_date|order_estimated_delivery_date|
+--------------------+--------------------+------------+------------------------+-------------------+----------------------------+-----------------------------+-----------------------------+
|cd0d42e029e3b56ba...|34a7113ee408811be...|   delivered|     2018-02-28 16:27:24|2018-03-02 02:15:51|         2018-03-02 20:08:13|          2018-04-07 14:24:39|          2018-03-20 00:00:00|
|85cecbd067fc6feb9...|ef200b7ad812c7883...|   delivered|     2017-08-18 22:30:29|2017-08-18 22:45:12|         2017-08-21 21:27:51|          2017-08-29 20:23:06|          2017-09-20 00:00:00|
|70cca242632c1f58a...|61892c983a579e842...|  

In [0]:
#filling missing data

orders_df_cleaned = orders_df.fillna({'order_delivered_customer_Date':'9999-12-31'})
orders_df_cleaned.show()

+--------------------+--------------------+------------+------------------------+-------------------+----------------------------+-----------------------------+-----------------------------+
|            order_id|         customer_id|order_status|order_purchase_timestamp|  order_approved_at|order_delivered_carrier_date|order_delivered_customer_date|order_estimated_delivery_date|
+--------------------+--------------------+------------+------------------------+-------------------+----------------------------+-----------------------------+-----------------------------+
|cd0d42e029e3b56ba...|34a7113ee408811be...|   delivered|     2018-02-28 16:27:24|2018-03-02 02:15:51|         2018-03-02 20:08:13|          2018-04-07 14:24:39|          2018-03-20 00:00:00|
|85cecbd067fc6feb9...|ef200b7ad812c7883...|   delivered|     2017-08-18 22:30:29|2017-08-18 22:45:12|         2017-08-21 21:27:51|          2017-08-29 20:23:06|          2017-09-20 00:00:00|
|70cca242632c1f58a...|61892c983a579e842...|  

# Impute missing values

In [0]:
#added null , to try Impute on that
payments_df_with_null = payments_df.withColumn('payment_value',when(col('payment_value')!= 99.33,col('payment_value')).otherwise(lit(None)))
payments_df_with_null.show()

+--------------------+------------------+------------+--------------------+-------------+
|            order_id|payment_sequential|payment_type|payment_installments|payment_value|
+--------------------+------------------+------------+--------------------+-------------+
|b81ef226f3fe1789b...|                 1| credit_card|                   8|         NULL|
|a9810da82917af2d9...|                 1| credit_card|                   1|        24.39|
|25e8ea4e93396b6fa...|                 1| credit_card|                   1|        65.71|
|ba78997921bbcdc13...|                 1| credit_card|                   8|       107.78|
|42fdf880ba16b47b5...|                 1| credit_card|                   2|       128.45|
|298fcdf1f73eb413e...|                 1| credit_card|                   2|        96.12|
|771ee386b001f0620...|                 1| credit_card|                   1|        81.16|
|3d7239c394a212faa...|                 1| credit_card|                   3|        51.84|
|1f78449c8

In [0]:
from pyspark.ml.feature import Imputer
imputer =Imputer(inputCols=['payment_value'],outputCols=['paymnet_value_imputed']).setStrategy('median')

payments_df_cleaned = imputer.fit(payments_df_with_null).transform(payments_df_with_null)  #we need to first fit the data and then transform
payments_df_cleaned.show()

+--------------------+------------------+------------+--------------------+-------------+---------------------+
|            order_id|payment_sequential|payment_type|payment_installments|payment_value|paymnet_value_imputed|
+--------------------+------------------+------------+--------------------+-------------+---------------------+
|b81ef226f3fe1789b...|                 1| credit_card|                   8|         NULL|                100.0|
|a9810da82917af2d9...|                 1| credit_card|                   1|        24.39|                24.39|
|25e8ea4e93396b6fa...|                 1| credit_card|                   1|        65.71|                65.71|
|ba78997921bbcdc13...|                 1| credit_card|                   8|       107.78|               107.78|
|42fdf880ba16b47b5...|                 1| credit_card|                   2|       128.45|               128.45|
|298fcdf1f73eb413e...|                 1| credit_card|                   2|        96.12|               

#Standardizing the format

In [0]:
def print_schema(df, df_name):
    print(f"schema of {df_name}")
    df.printSchema()



In [0]:
print_schema(customers_df,'customers_df')
print_schema(geolocation_df,'geolocation_df')
print_schema(order_items_df,'order_items_df')
print_schema(payments_df,'payments_df')
print_schema(reviews_df,'reviews_df')
print_schema(orders_df,'orders_df')
print_schema(products_df,'products_df')
print_schema(sellers_df,'sellers_df')
print_schema(catgeory_translation_df,'catgeory_translation_df')


schema of customers_df
root
 |-- customer_id: string (nullable = true)
 |-- customer_unique_id: string (nullable = true)
 |-- customer_zip_code_prefix: integer (nullable = true)
 |-- customer_city: string (nullable = true)
 |-- customer_state: string (nullable = true)

schema of geolocation_df
root
 |-- geolocation_zip_code_prefix: integer (nullable = true)
 |-- geolocation_lat: double (nullable = true)
 |-- geolocation_lng: double (nullable = true)
 |-- geolocation_city: string (nullable = true)
 |-- geolocation_state: string (nullable = true)

schema of order_items_df
root
 |-- order_id: string (nullable = true)
 |-- order_item_id: integer (nullable = true)
 |-- product_id: string (nullable = true)
 |-- seller_id: string (nullable = true)
 |-- shipping_limit_date: timestamp (nullable = true)
 |-- price: double (nullable = true)
 |-- freight_value: double (nullable = true)

schema of payments_df
root
 |-- order_id: string (nullable = true)
 |-- payment_sequential: integer (nullable = 

In [0]:
orders_df_cleaned = orders_df_cleaned.withColumn('order_purchase_timestamp',to_date(col('order_purchase_timestamp')))
orders_df_cleaned.show()

+--------------------+--------------------+------------+------------------------+-------------------+----------------------------+-----------------------------+-----------------------------+
|            order_id|         customer_id|order_status|order_purchase_timestamp|  order_approved_at|order_delivered_carrier_date|order_delivered_customer_date|order_estimated_delivery_date|
+--------------------+--------------------+------------+------------------------+-------------------+----------------------------+-----------------------------+-----------------------------+
|cd0d42e029e3b56ba...|34a7113ee408811be...|   delivered|              2018-02-28|2018-03-02 02:15:51|         2018-03-02 20:08:13|          2018-04-07 14:24:39|          2018-03-20 00:00:00|
|85cecbd067fc6feb9...|ef200b7ad812c7883...|   delivered|              2017-08-18|2017-08-18 22:45:12|         2017-08-21 21:27:51|          2017-08-29 20:23:06|          2017-09-20 00:00:00|
|70cca242632c1f58a...|61892c983a579e842...|  

In [0]:
payments_df.show(5)

+--------------------+------------------+------------+--------------------+-------------+
|            order_id|payment_sequential|payment_type|payment_installments|payment_value|
+--------------------+------------------+------------+--------------------+-------------+
|b81ef226f3fe1789b...|                 1| credit_card|                   8|        99.33|
|a9810da82917af2d9...|                 1| credit_card|                   1|        24.39|
|25e8ea4e93396b6fa...|                 1| credit_card|                   1|        65.71|
|ba78997921bbcdc13...|                 1| credit_card|                   8|       107.78|
|42fdf880ba16b47b5...|                 1| credit_card|                   2|       128.45|
+--------------------+------------------+------------+--------------------+-------------+
only showing top 5 rows


In [0]:
#Standardize the categorical data  

payments_df_cleaned = payments_df_cleaned.withColumn('payment_type',when(col('payment_type') == 'boleto','Bank Transfer').when(col('payment_type')=='credit_card','Credit Card').when(col('payment_type')=='debit_card','Debit Card').otherwise('other'))
payments_df_cleaned.show()

+--------------------+------------------+-------------+--------------------+-------------+---------------------+
|            order_id|payment_sequential| payment_type|payment_installments|payment_value|paymnet_value_imputed|
+--------------------+------------------+-------------+--------------------+-------------+---------------------+
|b81ef226f3fe1789b...|                 1|  Credit Card|                   8|         NULL|                100.0|
|a9810da82917af2d9...|                 1|  Credit Card|                   1|        24.39|                24.39|
|25e8ea4e93396b6fa...|                 1|  Credit Card|                   1|        65.71|                65.71|
|ba78997921bbcdc13...|                 1|  Credit Card|                   8|       107.78|               107.78|
|42fdf880ba16b47b5...|                 1|  Credit Card|                   2|       128.45|               128.45|
|298fcdf1f73eb413e...|                 1|  Credit Card|                   2|        96.12|      

In [0]:
customers_df.printSchema()

root
 |-- customer_id: string (nullable = true)
 |-- customer_unique_id: string (nullable = true)
 |-- customer_zip_code_prefix: integer (nullable = true)
 |-- customer_city: string (nullable = true)
 |-- customer_state: string (nullable = true)



In [0]:
# this customer_zip_code_prefix if inferred as string -- but we as humabn know that it shoul;d be string -- as it can be mistaken to be integer and any calculation can be done 
customers_df_cleaned = customers_df.withColumn('customer_zip_code_prefix',col('customer_zip_code_prefix').cast('string'))
customers_df_cleaned.printSchema()

root
 |-- customer_id: string (nullable = true)
 |-- customer_unique_id: string (nullable = true)
 |-- customer_zip_code_prefix: string (nullable = true)
 |-- customer_city: string (nullable = true)
 |-- customer_state: string (nullable = true)



##Remove Duplicate Records

In [0]:
customers_df_cleaned = customers_df_cleaned.dropDuplicates(["customer_id"])
customers_df_cleaned.show()

+--------------------+--------------------+------------------------+-------------------+--------------+
|         customer_id|  customer_unique_id|customer_zip_code_prefix|      customer_city|customer_state|
+--------------------+--------------------+------------------------+-------------------+--------------+
|e3c7e245a96d7fa33...|79051ee5ee98c4bd6...|                    7847|    franco da rocha|            SP|
|a56b03f5e6015f1a5...|b6cbe1a8674ee23e9...|                   41308|           salvador|            BA|
|d0615859a639a94c1...|9072b46e3b6896156...|                   12900|  braganca paulista|            SP|
|c0fe0fbc24994167d...|839bbfd4ff93b592c...|                   30330|     belo horizonte|            MG|
|5b5f4957a69d537a2...|bb03ed8d9549898e8...|                   71505|           brasilia|            DF|
|41b200d1ce8675f15...|9b05b38d7d9ef19a8...|                    2114|          sao paulo|            SP|
|456c1e01c8ed3b83a...|4ac6ec83ece04605a...|                   59

In [0]:
order_with_details = orders_df_cleaned.join(order_items_df,'order_id','left')\
    .join(payments_df_cleaned,'order_id','left')\
        .join(customers_df_cleaned,'customer_id','left')

In [0]:
order_with_details.display()

customer_id,order_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date,order_item_id,product_id,seller_id,shipping_limit_date,price,freight_value,payment_sequential,payment_type,payment_installments,payment_value,paymnet_value_imputed,customer_unique_id,customer_zip_code_prefix,customer_city,customer_state
34a7113ee408811be7f4bfc68651bde5,cd0d42e029e3b56ba5f53f6c59e5541b,delivered,2018-02-28,2018-03-02T02:15:51Z,2018-03-02T20:08:13Z,2018-04-07T14:24:39Z,2018-03-20T00:00:00Z,1,c695342500b74ff952db47b541726de1,1eade46fba20122dc4aefb379f8c636b,2018-03-08T02:15:51Z,11.99,14.1,1,Bank Transfer,1,52.18,52.18,3a4061148f13c4e70eb7f6047a8b01a2,29170,serra,ES
ef200b7ad812c7883186f2fb0482ebe0,85cecbd067fc6feb989de87d22126955,delivered,2017-08-18,2017-08-18T22:45:12Z,2017-08-21T21:27:51Z,2017-08-29T20:23:06Z,2017-09-20T00:00:00Z,1,601a360bd2a916ecef0e88de72a6531a,7a67c85e85bb2ce8582c35f2203ad736,2017-08-23T22:45:12Z,118.99,20.07,1,Credit Card,4,139.06,139.06,bbd65b8d05059eb992faa0208fab2e0d,46140,livramento de nossa senhora,BA
61892c983a579e842242d67db04c0925,70cca242632c1f58abe233c9ea071415,delivered,2018-04-06,2018-04-06T21:50:10Z,2018-04-09T13:04:55Z,2018-04-13T15:36:46Z,2018-04-26T00:00:00Z,1,d15768e35ed029db22ccd67cb15ec8eb,a56e351445e5863fc8960640ba9190c7,2018-04-11T21:50:10Z,184.22,20.26,1,Credit Card,1,204.48,204.48,21e292ef7f932a8e49aec645ac36e033,4052,sao paulo,SP
6d1e7c6a52ef57ca5462d78e575a0c1b,d063dcc119860db74b0e76cbac5fb4ca,delivered,2018-08-21,2018-08-21T01:15:12Z,2018-08-21T11:58:00Z,2018-08-29T20:48:33Z,2018-09-18T00:00:00Z,1,aa8d88eb4b9cb38894e33fa624c4287f,6560211a19b47992c3666cc44a7e94c0,2018-08-23T01:15:12Z,54.0,19.29,1,Credit Card,3,73.29,73.29,b6264382015251f0779ccb486bd9cc8e,51020,recife,PE
5b088866d5d40ceef628716f9ac1173f,ba436db3c441f63a8ed34e695ed2e9f4,delivered,2018-07-31,2018-08-01T04:35:11Z,2018-08-01T12:13:00Z,2018-08-03T17:22:54Z,2018-08-13T00:00:00Z,1,c4baedd846ed09b85f78a781b522f126,a1043bafd471dff536d0c462352beb48,2018-08-03T04:32:27Z,99.99,23.69,1,Bank Transfer,1,123.68,123.68,4483e2d9adf752a7b9f781eea1251bca,31842,belo horizonte,MG
2a612257efe760db38fa27d6e45f9d6f,eaa0211ae0667cb758713505ee149479,delivered,2017-08-21,2017-08-21T12:45:40Z,2017-08-23T20:28:56Z,2017-08-24T20:33:41Z,2017-09-05T00:00:00Z,1,eaecac42c42a65691bc3662b123e884d,916748bc99315c2d202898ae58b1617e,2017-08-25T12:45:40Z,15.0,8.27,2,other,1,10.03,10.03,b43b59147bb1d34fbeab3eba7b7fe372,1442,sao paulo,SP
caed95700dcb091eeaac59a97fe0cb17,82ffe097d8ddbf319a523b9bbe7725d5,delivered,2018-04-23,2018-04-24T18:29:30Z,2018-04-23T16:52:42Z,2018-04-24T17:34:30Z,2018-05-08T00:00:00Z,1,31cddbc370031fbb839a21441df28015,41e0fa5761c886a630994a55c12087e7,2018-04-27T02:31:45Z,84.6,8.89,2,other,1,30.0,30.0,587d22544f4d6f17b39a30723a53a305,8557,poa,SP
9cc68dabd982b73d1e5d270a58bb59d1,885ab07cf96454b41862d8b17e76a901,delivered,2018-01-11,2018-01-13T02:17:01Z,2018-01-15T21:18:50Z,2018-01-31T00:48:51Z,2018-02-15T00:00:00Z,1,eaa0c1e54b091f60f2f685b71d71f59a,8bb48dc19fccaa8613b6229bf7f452a2,2018-01-18T02:17:01Z,16.6,25.63,1,Bank Transfer,1,42.23,42.23,640492ff1e75a415ad4ac51fa0878a6b,62500,itapipoca,CE
92dc1467d2ba256b84a991c5a3218221,be13c8dcfef55266122ed7d751f92ce3,delivered,2018-04-20,2018-04-25T04:11:22Z,2018-04-25T13:38:00Z,2018-04-30T19:24:42Z,2018-05-15T00:00:00Z,1,79da264732f717f10ebf5d102aa6c32a,562fc2f2c2863ab7e79a9e4388a58a14,2018-05-02T04:11:22Z,29.99,12.79,1,Bank Transfer,1,42.78,42.78,4bb56718dde10c146405afbebf7b9454,18212,itapetininga,SP
d8d4bfa3aa3d6a7fa3760a50e532383f,1aabc8c7c8525b6ada34a6a07d88e26d,delivered,2018-04-23,2018-04-24T19:09:53Z,2018-04-25T15:07:00Z,2018-04-26T22:32:37Z,2018-05-14T00:00:00Z,1,3a5bfc952a6ad85598f16e431eb7622b,23c38debaffe4a25a30fdbd9b586a13f,2018-05-02T17:31:50Z,289.0,14.46,1,Credit Card,2,303.46,303.46,5711423ece621907bba6bdc5178c0f3e,7600,mairipora,SP


In [0]:
order_With_total_value = order_with_details.groupBy('order_id')\
    .agg(sum('payment_value').alias('total_order_value'))

In [0]:
order_With_total_value.show()

+--------------------+-----------------+
|            order_id|total_order_value|
+--------------------+-----------------+
|92fc1ca41d579ec86...|            56.38|
|0b0b271dbd163df37...|           108.86|
|4609e3dffa7470680...|           853.82|
|db7059aeda4a4b33b...|            32.05|
|add0a7ea74e304802...|            51.33|
|41537821ce113ccef...|            89.27|
|6a29f292d5b7ce8ed...|           130.27|
|077700dcf4e3bb412...|           215.05|
|387ab69cd89bb3ed9...|            103.6|
|618a75a5455f1971e...|           418.04|
|ce8f65c6a6f857ba8...|            65.13|
|ca98a7276eda33f33...|           249.76|
|e239d280236cdd3c4...|           102.03|
|3f090361c1e3266bf...|            28.88|
|7dc8379a32999985a...|            86.55|
|ea70810a7aa753a29...|            40.23|
|ac086c46cd97732d9...|           110.98|
|12052ac6aa0674343...|           117.57|
|3e5d60bbe5a6016db...|            22.75|
|e4e28f77bd50442ca...|           125.77|
+--------------------+-----------------+
only showing top

In [0]:
#Delivery time calculation from previous notebook - [2-DataExploration] 
#is also feature engineering

#Advance Transformations


Search - outlier treatment boxplot = Interquartilerange
--- yes i GUES data with Baraa - in some video i saw == in PYTHON  while learning

#1
Outliers are abnormally high or low values that can distort analysis, averages, ML models, and business insights.

Using quantiles/percentiles, we define a normal range.

Values outside the range:
price < low_cutoff OR price > high_cutoff
are treated as outliers.

To clean data, we keep only values inside the cutoff range.

#2
You heard:

Interquartile Range (IQR)

That is ANOTHER common outlier detection method.

Uses:

Q1
Q3
IQR = Q3 - Q1

Very common in:

boxplots
statistics
pandas/python

Your current method:

percentile cutoff method

is simpler and scalable for Spark big data.

#3About skewness

YES — instructor meant:

A) Huge values pull:

average
distribution
graph

towards one side.

That is:

skewed data

B)when Lots of 0s - this will imapct teh data -- so hsould get rid of 

In [0]:
order_items_df.show()



+--------------------+-------------+--------------------+--------------------+-------------------+-----+-------------+
|            order_id|order_item_id|          product_id|           seller_id|shipping_limit_date|price|freight_value|
+--------------------+-------------+--------------------+--------------------+-------------------+-----+-------------+
|88498659c4bfb6880...|            1|24ef70f010919148b...|a938325a4b357fd23...|2018-03-07 17:30:31| 89.0|        29.02|
|884989e29f137d8ce...|            1|fd76f22bf3648e0dd...|6d66611d7c44cc30c...|2018-07-11 16:31:30| 39.9|        13.86|
|8849e8100a1269ca2...|            1|1cd35eaf33a642d4d...|cac4c8e7b1ca6252d...|2017-07-11 17:24:16|49.99|         16.6|
|8849f02bf66502f56...|            1|5ae2f57d379f89f99...|d594982fd877af63a...|2018-08-08 03:45:33|27.19|        18.29|
|884b1394fc8888e6a...|            1|437c05a395e9e47f9...|f84fa566034f5e8e8...|2018-03-25 22:08:28|47.65|         7.39|
|884b42885f11cac2a...|            1|a44f1df44a1a

In [0]:
## to remove the outliers == means some value can be worng -

#removing outlier from price
quantiles = order_items_df.approxQuantile("price",[0.01,0.99],0.0)
low_cutoff, high_cutoff = quantiles[0],quantiles[1]

In [0]:
low_cutoff, high_cutoff

(9.99, 890.0)

In [0]:
order_items_df.select('price').summary().show()   ## to view the summary of this column - to verify

+-------+-----------------+
|summary|            price|
+-------+-----------------+
|  count|           112650|
|   mean|120.6537390146417|
| stddev|183.6339280502595|
|    min|             0.85|
|    25%|             39.9|
|    50%|            74.99|
|    75%|            134.9|
|    max|           6735.0|
+-------+-----------------+



In [0]:
order_items_df_cleaned = order_items_df.filter((col('price') >=low_cutoff) & (col('price') <=high_cutoff))
order_items_df_cleaned.show()

+--------------------+-------------+--------------------+--------------------+-------------------+-----+-------------+
|            order_id|order_item_id|          product_id|           seller_id|shipping_limit_date|price|freight_value|
+--------------------+-------------+--------------------+--------------------+-------------------+-----+-------------+
|88498659c4bfb6880...|            1|24ef70f010919148b...|a938325a4b357fd23...|2018-03-07 17:30:31| 89.0|        29.02|
|884989e29f137d8ce...|            1|fd76f22bf3648e0dd...|6d66611d7c44cc30c...|2018-07-11 16:31:30| 39.9|        13.86|
|8849e8100a1269ca2...|            1|1cd35eaf33a642d4d...|cac4c8e7b1ca6252d...|2017-07-11 17:24:16|49.99|         16.6|
|8849f02bf66502f56...|            1|5ae2f57d379f89f99...|d594982fd877af63a...|2018-08-08 03:45:33|27.19|        18.29|
|884b1394fc8888e6a...|            1|437c05a395e9e47f9...|f84fa566034f5e8e8...|2018-03-25 22:08:28|47.65|         7.39|
|884b42885f11cac2a...|            1|a44f1df44a1a

In [0]:
payments_df_cleaned.show()

+--------------------+------------------+-------------+--------------------+-------------+---------------------+
|            order_id|payment_sequential| payment_type|payment_installments|payment_value|paymnet_value_imputed|
+--------------------+------------------+-------------+--------------------+-------------+---------------------+
|b81ef226f3fe1789b...|                 1|  Credit Card|                   8|         NULL|                100.0|
|a9810da82917af2d9...|                 1|  Credit Card|                   1|        24.39|                24.39|
|25e8ea4e93396b6fa...|                 1|  Credit Card|                   1|        65.71|                65.71|
|ba78997921bbcdc13...|                 1|  Credit Card|                   8|       107.78|               107.78|
|42fdf880ba16b47b5...|                 1|  Credit Card|                   2|       128.45|               128.45|
|298fcdf1f73eb413e...|                 1|  Credit Card|                   2|        96.12|      

In [0]:
payments_df_cleaned.select('payment_installments').summary().show()

+-------+--------------------+
|summary|payment_installments|
+-------+--------------------+
|  count|              103886|
|   mean|   2.853348863176944|
| stddev|   2.687050673856478|
|    min|                   0|
|    25%|                   1|
|    50%|                   1|
|    75%|                   4|
|    max|                  24|
+-------+--------------------+



In [0]:
#if we want to reduce the installments - if requirement iis that -----> then we can CAP the value  - so instead of removing the row ->cap it

paymnets_df_capped = payments_df_cleaned.withColumn("payment_installments", when(col("payment_installments")>12, 12).otherwise(col("payment_installments")))
paymnets_df_capped.select('payment_installments').summary().show()

+-------+--------------------+
|summary|payment_installments|
+-------+--------------------+
|  count|              103886|
|   mean|  2.8447817800281077|
| stddev|  2.6465754501526773|
|    min|                   0|
|    25%|                   1|
|    50%|                   1|
|    75%|                   4|
|    max|                  12|
+-------+--------------------+



##Featue Engineering

In [0]:
products_df.show()

+--------------------+---------------------+-------------------+--------------------------+------------------+----------------+-----------------+-----------------+----------------+
|          product_id|product_category_name|product_name_lenght|product_description_lenght|product_photos_qty|product_weight_g|product_length_cm|product_height_cm|product_width_cm|
+--------------------+---------------------+-------------------+--------------------------+------------------+----------------+-----------------+-----------------+----------------+
|b316bc5e17977e45d...| informatica_acess...|                 45|                       311|                 2|             300|               20|               20|              20|
|8a330ba3cd34d650c...|           cool_stuff|                 44|                      1611|                 2|            2400|               32|               25|              39|
|e3025f416b18907e0...|         beleza_saude|                 60|                      1040|    

In [0]:
#feature engineering - to get products overall what is teh size -- meduim large

products_df_cleaned = products_df.withColumn("product_size",when(col('product_weight_g')<500,'Small')
                                             .when(col("product_weight_g").between(500,2000),'Medium')
                                             .otherwise('Large')
                                             )
products_df_cleaned.show()                                         

+--------------------+---------------------+-------------------+--------------------------+------------------+----------------+-----------------+-----------------+----------------+------------+
|          product_id|product_category_name|product_name_lenght|product_description_lenght|product_photos_qty|product_weight_g|product_length_cm|product_height_cm|product_width_cm|product_size|
+--------------------+---------------------+-------------------+--------------------------+------------------+----------------+-----------------+-----------------+----------------+------------+
|b316bc5e17977e45d...| informatica_acess...|                 45|                       311|                 2|             300|               20|               20|              20|       Small|
|8a330ba3cd34d650c...|           cool_stuff|                 44|                      1611|                 2|            2400|               32|               25|              39|       Large|
|e3025f416b18907e0...|        

In [0]:
#Total revenue per seller

sellers_df.show()

+--------------------+----------------------+-----------------+------------+
|           seller_id|seller_zip_code_prefix|      seller_city|seller_state|
+--------------------+----------------------+-----------------+------------+
|3442f8959a84dea7e...|                 13023|         campinas|          SP|
|d1b65fc7debc3361e...|                 13844|       mogi guacu|          SP|
|ce3ad9de960102d06...|                 20031|   rio de janeiro|          RJ|
|c0f3eea2e14555b6f...|                  4195|        sao paulo|          SP|
|51a04a8a6bdcb23de...|                 12914|braganca paulista|          SP|
|c240c4061717ac180...|                 20920|   rio de janeiro|          RJ|
|e49c26c3edfa46d22...|                 55325|           brejao|          PE|
|1b938a7ec6ac5061a...|                 16304|        penapolis|          SP|
|768a86e36ad6aae3d...|                  1529|        sao paulo|          SP|
|ccc4bbb5f32a6ab2b...|                 80310|         curitiba|          PR|